# Bagian 1: Penjelasan Konsep

## 1.1 Peta Besar: Method Estimator

```text
                    Estimator (semua punya .fit())
                    /                          \
            Transformer                    Predictor
        (.fit + .transform)           (.fit + .predict)
                                              |
                                    ┌─────────┴─────────┐
                                Classifier            Regressor
                          (punya .predict_proba,    (predict_proba
                           .decision_function)        TIDAK ada)
```

* **Predictor** (model) selalu memiliki `.predict()`.
* **Classifier** biasanya memiliki method tambahan:

  * `.predict_proba()` → menghasilkan probabilitas setiap kelas.
  * `.decision_function()` → menghasilkan nilai keputusan/skor untuk klasifikasi.
* **Regressor** tidak menggunakan kedua method tersebut karena outputnya berupa **nilai kontinu**, bukan probabilitas atau kelas.
* Jadi secara sederhana:

  * **Classifier** → `.predict()`, `.predict_proba()`, `.decision_function()`
  * **Regressor** → `.predict()`


## 1.2 .fit() — Method Fundamental

```python
estimator.fit(X, y=None)
```

* **Transformer** — contoh: `StandardScaler`, `OneHotEncoder`

  * Menggunakan `fit(X)`.
  * Hanya membutuhkan **fitur (`X`)**.
  * Tujuannya mempelajari statistik atau struktur dari data.
  * Tidak bertugas memprediksi target.

* **Predictor supervised** — contoh: `LogisticRegression`

  * Menggunakan `fit(X, y)`.
  * Membutuhkan **fitur (`X`) dan label (`y`)**.
  * Tujuannya mempelajari hubungan antara `X` dan `y`.

* **Model unsupervised** — contoh: `KMeans`

  * Menggunakan `fit(X)`.
  * Hanya membutuhkan **fitur (`X`)**.
  * Tidak membutuhkan `y` karena tidak terdapat label yang harus dipelajari.


* **`.fit()` selalu mengembalikan `self`**, yaitu objek estimator itu sendiri.
* `.fit()` **bukan** method untuk menghasilkan transformasi atau prediksi.
* Karena mengembalikan objek yang sama, method dapat langsung dirangkai:

```python
scaler = StandardScaler().fit(X_train)
```

Secara konsep:

```text
StandardScaler()
      ↓
    .fit(X)
      ↓
  self (scaler)
```

Jadi:

```python
scaler = StandardScaler().fit(X_train)
```

setara dengan:

```python
scaler = StandardScaler()
scaler.fit(X_train)
```

**Intinya:** `.fit()` → **belajar dari data → mengembalikan estimator yang sudah di-fit**.


```python
scaler = StandardScaler()
print(scaler.fit(X_train))
# Output: StandardScaler()   ← ini objeknya sendiri, bukan datanya

##  1.3 .transform() vs .fit_transform()

* `.transform(X)` → menerapkan aturan/statistik yang sudah dipelajari melalui `.fit()` sebelumnya.
* `.fit_transform(X)` → melakukan dua langkah sekaligus:

  1. Mempelajari aturan/statistik dari `X`.
  2. Langsung menerapkan hasil belajar tersebut pada `X`.

Secara konsep:

```python
transformer.fit(X)
X_transformed = transformer.transform(X)
```

setara dengan:

```python
X_transformed = transformer.fit_transform(X)
```

Intinya:

* `.fit()` → belajar
* `.transform()` → menerapkan hasil belajar
* `.fit_transform()` → belajar + menerapkan


* Untuk beberapa transformer, `.fit_transform()` dapat lebih efisien secara komputasi dibandingkan memanggil `.fit()` dan `.transform()` secara terpisah.
* Hal ini karena implementasi transformer tertentu dapat mengoptimalkan perhitungan antara proses belajar dan transformasi.
* Hasil akhirnya tetap sama secara konsep.
* Perbedaannya terletak pada **efisiensi proses di balik layar**, bukan pada hasil transformasinya.

Intinya:

```text
Training data → fit_transform()   (belajar DAN terapkan, sekaligus)
Testing data  → transform() SAJA   (terapkan HASIL BELAJAR dari training, JANGAN belajar lagi)
```


```pyhton
X_train_scaled = scaler.fit_transform(X_train)   # ✅ fit + transform di train
X_test_scaled = scaler.transform(X_test)          # ✅ transform SAJA di test
```

* Jangan menggunakan `scaler.fit_transform(X_test)`.
* `fit_transform()` akan:

  1. Menghitung mean dan standard deviation dari `X_test`.
  2. Menggunakan statistik tersebut untuk mentransformasi `X_test`.
* Ini menyebabkan scaler **belajar dari data test**.
* Seharusnya, statistik preprocessing hanya dipelajari dari `X_train`.
* Data test hanya boleh menggunakan hasil belajar tersebut:

```python
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

Intinya:

```text
X_train → fit + transform
X_test  → transform saja
```

Jika `fit_transform(X_test)` digunakan, informasi dari test ikut masuk ke proses preprocessing → **data leakage**.


## 1.4 predict() — Method untuk Predictor

```python
model.predict(X_new)

Method ini hanya tersedia setelah .fit() dipanggil, dan mengembalikan hasil prediksi akhir:
* Untuk classifier: label kelas (misal 0, 1, 2, atau string seperti "churn"/"no_churn")
* Untuk regressor: angka kontinu (misal prediksi harga rumah 450000000)

```python
model = LogisticRegression()
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(predictions[:5])   # Output: array([0, 1, 0, 2, 1])  ← label kelas langsung
```

## 1.5 predict_proba() — Probabilitas di Balik Prediksi

Ini method khusus classifier (tidak ada di regressor). Alih-alih langsung memberi label akhir, method ini memberi tahu seberapa yakin model terhadap tiap kemungkinan kelas.

```python
probs = model.predict_proba(X_test)
print(probs[:3])
# Output contoh:
# [[0.81, 0.15, 0.04],   ← baris 1: 81% yakin kelas 0, 15% kelas 1, 4% kelas 2
#  [0.02, 0.90, 0.08],   ← baris 2: 90% yakin kelas 1
#  [0.33, 0.30, 0.37]]   ← baris 3: model "ragu-ragu", hampir seimbang

* Regressor

  * Prediksi dilakukan menggunakan `predict()`.
  * Output `predict()` merupakan nilai prediksi akhir.

* Classifier

  * Biasanya menggunakan `predict_proba()` untuk menghasilkan probabilitas setiap kelas.
  * Probabilitas tersebut kemudian digunakan untuk menentukan keputusan akhir.
  * Keputusan akhir inilah yang dikembalikan oleh `predict()`.
  * Jadi, secara sederhana: `predict_proba()` → probabilitas → `predict()` → kelas akhir.


Artinya:

* `model.predict()` pada classifier sebenarnya merupakan "turunan" dari `predict_proba()`.
* Di belakang layar, model menghitung probabilitas untuk setiap kelas.
* Scikit-learn kemudian memilih kelas dengan probabilitas tertinggi.
* Kelas dengan probabilitas tertinggi tersebut dikembalikan sebagai hasil `model.predict()`.


* `predict()` hanya memberikan keputusan akhir, misalnya:

  * `0` → tidak fraud
  * `1` → fraud

* `predict_proba()` memberikan probabilitas untuk setiap kelas.

  * Contoh: `P(tidak fraud) = 0.25`
  * `P(fraud) = 0.75`

* `predict_proba()` berguna ketika kita ingin mengetahui seberapa yakin model terhadap prediksinya.

* Contoh pada fraud detection:

  * Probabilitas fraud > 70% → tandai sebagai perlu review manual.
  * Probabilitas fraud ≤ 70% → tidak perlu review.

* Jadi:

  * `predict()` → apa keputusan model?
  * `predict_proba()` → seberapa yakin model terhadap keputusan tersebut?


> Catatan: tidak semua classifier punya predict_proba() secara native — beberapa (seperti SVM tertentu) menggunakan decision_function() sebagai gantinya, yang mengembalikan skor mentah (bukan probabilitas 0–1). Topik ini di luar cakupan mendalam materi ini, tapi penting untuk kamu tahu bahwa method ini ada.

## 1.6 score() — Evaluasi Cepat Bawaan

Hampir semua predictor punya method .score() yang memberi angka evaluasi instan tanpa kamu perlu import fungsi metrik terpisah.

```python
akurasi = model.score(X_test, y_test)

* Metrik default pada `score()` berbeda tergantung jenis model. ([Scikit-learn][1])

* Classifier

  * Default `score()` umumnya menggunakan `accuracy`.
  * Mengukur seberapa banyak prediksi kelas yang benar.

* Regressor

  * Default `score()` umumnya menggunakan koefisien determinasi `R²`.
  * Mengukur seberapa baik model menjelaskan variasi pada target.

* Jadi, secara sederhana:

  * Classifier → `model.score()` → `accuracy`
  * Regressor → `model.score()` → `R²`

* Penting: `score()` bukan berarti selalu menggunakan metrik yang paling tepat untuk kasus bisnis.

  * Untuk kasus tertentu, kita dapat menentukan metrik lain menggunakan `scoring`, misalnya `f1`, `recall`, `roc_auc`, `MAE`, atau `RMSE`. ([Scikit-learn][1])

[1]: https://scikit-learn.org/stable/modules/model_evaluation.html?utm_source=chatgpt.com "3.4. Metrics and scoring: quantifying the quality of predictions — scikit-learn 1.9.0 documentation"


| Jenis Model                             | Default `.score()`                | Mengukur Apa                                    |
| --------------------------------------- | --------------------------------- | ----------------------------------------------- |
| Classifier (misal `LogisticRegression`) | Accuracy                          | Proporsi prediksi yang benar                    |
| Regressor (misal `LinearRegression`)    | R² (coefficient of determination) | Seberapa baik model menjelaskan variansi target |


* Angka `.score()` harus diinterpretasikan berdasarkan jenis model.
* Classifier:

  * `.score() = 0.85` → sekitar 85% prediksi benar (accuracy).
* Regressor:

  * `.score() = 0.85` → model menjelaskan sekitar 85% variansi pada target (R²).
* Jadi, angka yang sama tidak selalu memiliki makna yang sama:

  * `0.85` pada classifier → accuracy 85%.
  * `0.85` pada regressor → R² = 0.85.


> .score() itu praktis untuk cek cepat, tapi di dunia kerja nyata kamu hampir selalu butuh metrik yang lebih spesifik sesuai konteks bisnis (precision, recall, F1, RMSE, dsb.) — ini akan dibahas mendalam di materi 3.5 Evaluasi Model. Anggap .score() sebagai "sekilas info", bukan evaluasi final.

1.7 Ringkasan Perbandingan Semua Method

| Method             | Tersedia di         | Input                           | Output                       | Kapan dipakai                                                 |
| ------------------ | ------------------- | ------------------------------- | ---------------------------- | ------------------------------------------------------------- |
| `fit(X, y)`        | Semua estimator     | Fitur (+ label jika supervised) | `self`                       | Fase belajar                                                  |
| `transform(X)`     | Transformer         | Fitur                           | Data yang sudah diubah       | Terapkan hasil belajar ke data, biasanya test                 |
| `fit_transform(X)` | Transformer         | Fitur                           | Data yang sudah diubah       | Belajar + terapkan sekaligus, biasanya train                  |
| `predict(X)`       | Predictor           | Fitur                           | Label kelas / angka prediksi | Menghasilkan prediksi akhir                                   |
| `predict_proba(X)` | Classifier tertentu | Fitur                           | Probabilitas tiap kelas      | Saat membutuhkan tingkat keyakinan, bukan hanya jawaban akhir |
| `score(X, y)`      | Predictor           | Fitur + label asli              | Satu angka (accuracy/R²)     | Mengecek performa secara cepat                                |


# Bagian 2: Implementasi

## 2.1 Setup

In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import pandas as pd

## 2.2 Load dataset

In [2]:
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="species")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## 2.3 Membuktikan fit_transform() Setara dengan fit() + transform()

In [3]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Cara 1: fit_transform langsung
scaler_a = StandardScaler()
X_a = scaler_a.fit_transform(X_train)

# Cara 2: fit lalu transform terpisah
scaler_b = StandardScaler()
scaler_b.fit(X_train)
X_b = scaler_b.transform(X_train)

# Bandingkan — harus identik
print("Hasil identik?", np.allclose(X_a, X_b))
print("Mean identik?", np.allclose(scaler_a.mean_, scaler_b.mean_))

Hasil identik? True
Mean identik? True


## 2.4 predict() vs predict_proba() — Perbandingan Langsung

In [4]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

pred_label = model.predict(X_test)
pred_proba = model.predict_proba(X_test)

print("Kelas yang dikenali model:", model.classes_)
print("\n5 prediksi pertama (label):", pred_label[:5])
print("\n5 prediksi pertama (probabilitas per kelas):")
print(pred_proba[:5])

# Buktikan: predict() = argmax dari predict_proba()
label_dari_proba = pred_proba.argmax(axis=1)
print("\nApakah predict() = argmax(predict_proba())?",
      np.array_equal(pred_label, label_dari_proba))

Kelas yang dikenali model: [0 1 2]

5 prediksi pertama (label): [0 2 1 1 0]

5 prediksi pertama (probabilitas per kelas):
[[9.85319970e-01 1.46799833e-02 4.62310054e-08]
 [1.37591104e-03 3.90976485e-01 6.07647604e-01]
 [1.86892614e-01 8.08902695e-01 4.20469116e-03]
 [1.55874534e-01 8.39893732e-01 4.23173454e-03]
 [9.88258461e-01 1.17415040e-02 3.53987834e-08]]

Apakah predict() = argmax(predict_proba())? True


## 2.5 score() — Classifier vs Regressor

In [5]:
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.datasets import fetch_california_housing

# --- Classifier: score() = accuracy ---
clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)
print("Classifier .score() [accuracy]:", clf.score(X_test, y_test))

# --- Regressor: score() = R² ---
housing = fetch_california_housing()
X_house = pd.DataFrame(housing.data, columns=housing.feature_names)
y_house = pd.Series(housing.target)

X_h_train, X_h_test, y_h_train, y_h_test = train_test_split(
    X_house, y_house, test_size=0.2, random_state=42
)

reg = LinearRegression()
reg.fit(X_h_train, y_h_train)
print("Regressor .score() [R²]:", reg.score(X_h_test, y_h_test))

Classifier .score() [accuracy]: 0.9666666666666667
Regressor .score() [R²]: 0.5757877060324511


Perhatikan: kode struktur-nya identik (fit() lalu score()), tapi makna angkanya berbeda total — inilah kenapa memahami konteks jenis model itu penting, bukan cuma menjalankan kode secara mekanis.

## 2.6 Workflow Gabungan — Semua Method dalam Satu Alur

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Preprocessing
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit_transform di train
X_test_scaled = scaler.transform(X_test)          # transform saja di test

# 2. Modeling
model = LogisticRegression(max_iter=200)
model.fit(X_train_scaled, y_train)                # fit model HANYA di train

# 3. Prediksi
y_pred = model.predict(X_test_scaled)              # predict di test
y_pred_proba = model.predict_proba(X_test_scaled)   # predict_proba di test

# 4. Evaluasi — dua cara, harus identik
print("Akurasi via .score():", model.score(X_test_scaled, y_test))
print("Akurasi via accuracy_score():", accuracy_score(y_test, y_pred))

Akurasi via .score(): 0.9333333333333333
Akurasi via accuracy_score(): 0.9333333333333333


> .score() dan accuracy_score() (dari sklearn.metrics) harus menghasilkan angka yang sama untuk classifier — karena .score() secara default memanggil accuracy_score() di balik layar.

## Kesalahan Umum

| Kesalahan                                                                                  | Dampaknya                                                                                                                      |
| ------------------------------------------------------------------------------------------ | ------------------------------------------------------------------------------------------------------------------------------ |
| Memanggil `fit_transform()` pada data test                                                 | Data leakage — statistik dari test ikut "bocor" ke proses preprocessing                                                        |
| Mengira `predict()` dan `predict_proba()` selalu memberikan hasil dengan arti yang berbeda | `predict()` pada classifier umumnya memilih kelas dengan probabilitas tertinggi dari hasil `predict_proba()`                   |
| Menginterpretasikan angka `.score()` sama untuk classifier dan regressor                   | Accuracy (0–1, semakin tinggi semakin baik) ≠ R² (dapat bernilai negatif dan mengukur proporsi variansi yang dijelaskan model) |
| Memanggil `.transform()` sebelum `.fit()`                                                  | Akan error karena transformer belum mempelajari statistik yang diperlukan                                                      |
| Menganggap semua classifier memiliki `predict_proba()`                                     | Beberapa model menggunakan `decision_function()` sebagai alternatif — selalu cek dokumentasi model                             |
| Hanya mengandalkan `.score()` untuk evaluasi final pada project nyata                      | `.score()` hanya menggunakan metrik default; konteks bisnis sering membutuhkan metrik lain                                     |


# Bagian 3: Latihan Praktik

```text
Latihan 1 — Buktikan Kesetaraan fit_transform
Jalankan kode di bagian 2.2 dengan transformer lain, misalnya MinMaxScaler (dari sklearn.preprocessing). Apakah hasilnya tetap identik antara fit_transform() langsung vs fit()+transform() terpisah?

Latihan 2 — Eksplorasi predict_proba()
Dari kode di bagian 2.3, cari baris data di X_test di mana model paling tidak yakin (probabilitas tertinggi di predict_proba() paling mendekati merata/seimbang antar kelas). Tulis observasimu: menurutmu kenapa model "ragu-ragu" untuk baris data tersebut?

Latihan 3 — Bandingkan .score() dengan Model Lain
Ganti LogisticRegression di bagian 2.4 dengan DecisionTreeClassifier untuk kasus classifier, dan Ridge (dari sklearn.linear_model) untuk kasus regressor. Apakah pola .score() (accuracy untuk classifier, R² untuk regressor) tetap konsisten?

Latihan 4 — Coba Panggil transform() Sebelum fit()
Buat StandardScaler baru, langsung panggil .transform(X_train) tanpa .fit() dulu. Catat pesan error yang muncul, dan jelaskan dengan kata-katamu sendiri kenapa error itu masuk akal berdasarkan konsep yang sudah dipelajari.

Latihan 5 — Refleksi Tertulis
Tanpa membuka catatan, jawab singkat:

1. Kenapa fit_transform() di test data itu berbahaya, padahal secara sintaks tidak akan error?
2. Apa hubungan antara predict() dan predict_proba()?
3. Kalau kamu dapat .score() = 0.6 dari sebuah regressor, apa artinya itu (bukan "60% benar" seperti classifier)?